### Top Trend and graph saving

In [25]:
import pandas
from pathlib import Path
from sts.mktdata.ticker import Ticker
from sts.quant.candle import Candle
from datetime import date, timedelta, datetime
from pathlib import Path
import numpy
from sts.quant.volatility import atr
import traceback
ticker = Ticker('us/etf.yml')

In [22]:
trend_stats_folder = Path('~/data/stsdata/research/us_etf/trend_stats/1W').expanduser()
td = date.today()
top_symbols = ticker.top_trading_symbols(td)
dates = sorted([f.name for f in trend_stats_folder.iterdir()])
all_df = []
for d in dates[-1:]:
    df = pandas.read_parquet(trend_stats_folder / d)
    if df is not None and not df.empty:
        df['date'] = d
        all_df.append(df)
all_df = pandas.concat(all_df)[::-1]

Wait expired, Browser is being closed by watchdog.


In [ ]:
import plotly.io as pio
pio.renderers.default = "png"
day_freq = 252 / 5
vol_threshold = 50
skipped_symbols = []
image_folder = Path('~/stsinvest/top_trends/').expanduser()
for f in image_folder.iterdir():
    if f.is_file():
        f.unlink()
all_df['asof'] = all_df['asof'].astype(str)
all_df.to_excel(image_folder / f"trend_stats_{td}.xlsx", index = False)
for idx, row in all_df.iterrows():
    try:
        symbol = row['symbol']
        start = td - timedelta(days = 365 * 3)
        end = td
        df = ticker.history(symbol, start, end, interval = '1W').set_index('ts')
        df = Candle(df)
        fig = df.plot()
        vol = atr(df.log(), window = 20).iloc[-1] * 100.0 * numpy.sqrt(day_freq)
        if (vol < vol_threshold) or (vol > 200):
            skipped_symbols.append(symbol)
            continue
        fig.update_layout(title = f"{row.symbol}, trendScore={row['trendScore']:.3f}, vol={vol:.2f}%")
        score = f"{int(abs(row['trendScore']) * 100):03d}"
        score = ('P' + score) if row['trendScore'] > 0 else ('N' + score)
        pio.write_image(fig, str(image_folder/f"Score_{score}_{symbol}.png"), width=800, height=600, scale=2)
    except Exception as e:
        traceback.print_exc()

/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/1W/20260215.parquet does not exist, skip

KeyboardInterrupt: 

Wait expired, Browser is being closed by watchdog.


NameError: name 'score' is not defined

### Trend Study

In [11]:
import pandas
from pathlib import Path

In [12]:
trend_stats_folder = Path('~/data/stsdata/research/us_etf/trend_stats/1d').expanduser()
dates = [f.name for f in trend_stats_folder.iterdir()]

In [13]:
symbols = ['SLV', 'GLD', 'SPY', 'QQQ', 'EEM', 'IWM', 'XLY', 'XLF', 'XLI', 'XLK', 'XLE', 'XLV', 'XLB', 'XLU', 'XLP']
all_df = []
for td in dates:
    df = pandas.read_parquet(trend_stats_folder / td, filters=[('symbol', 'in', symbols)])
    if df is not None and not df.empty:
        df['date'] = td
        all_df.append(df)
all_df = pandas.concat(all_df)

In [14]:
tmp = all_df[all_df['symbol'].isin(symbols)].copy()
try:
    tmp['ts'] = tmp['ts_1w'].fillna(0) + tmp['trendScore'].fillna(0)
except Exception:
    tmp['ts'] = tmp['trendScore']
tmp = tmp.pivot_table(index = 'date', columns = 'symbol', values = 'ts') 
pandas.options.plotting.backend = "plotly"
tmp.plot(title = 'ETF Trend Scores Over Time')

In [15]:
tmp

symbol,EEM,GLD,IWM,QQQ,SLV,SPY,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY
date,,,,,,,,,,,,,,,
20190102,-0.200570,0.929654,-0.519752,-0.335604,0.948878,-0.452559,-0.236404,-0.548678,-0.372246,-0.452392,-0.356314,-0.760752,-0.552148,-0.404588,-0.276810
20190103,-0.261297,1.021254,-0.506246,-0.385517,1.056577,-0.484626,-0.286309,-0.484262,-0.375299,-0.486114,-0.460533,-0.737226,-0.571646,-0.455119,-0.285460
20190104,-0.115444,0.898628,-0.323633,-0.250889,1.092145,-0.334322,-0.145746,-0.314406,-0.228124,-0.340241,-0.362872,-0.575836,-0.481034,-0.336120,-0.150349
20190107,-0.018515,0.876256,-0.140732,-0.129060,1.059915,-0.208402,-0.047890,-0.155082,-0.132670,-0.218678,-0.275391,-0.474305,-0.456621,-0.247161,0.019301
20190108,0.063764,0.799884,0.037185,-0.015832,1.033926,-0.082017,0.057991,-0.027133,-0.066029,-0.081493,-0.184509,-0.349990,-0.357068,-0.150504,0.166522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20260203,0.690249,0.355392,0.135052,0.014975,-0.000090,0.022126,0.677960,0.898110,-0.223793,0.599562,-0.114042,0.965623,0.049436,-0.208565,-0.052117
20260204,0.492872,0.314906,0.066455,-0.181263,-0.058160,0.002187,0.855006,1.036792,-0.142145,0.653206,-0.328368,1.150155,0.042689,-0.125371,-0.127339
20260205,0.336583,0.193072,-0.087657,-0.388954,-0.279834,-0.037841,0.718121,1.007066,-0.188779,0.624200,-0.536528,1.221825,0.041113,-0.135910,-0.308773
